# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook loads and explores the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is provided as a FAIR Croissant package and contains structured survey responses for mental health indicators, demographics, and assessment scores (GAD-7, PHQ-9, PSQ).

### Dataset Source
- **Croissant schema URL**: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
meta = dataset.metadata
print(f"{getattr(meta, 'name', 'N/A')}: {getattr(meta, 'description', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")

print("\nAvailable attributes in metadata:")
for attr in dir(meta):
    if not attr.startswith('_') and not callable(getattr(meta, attr)):
        print(f" - {attr}")

## 2. Data Overview

Review available **record sets**, their `@id`s, and fields, in order to plan extraction and analysis. We'll use the `list_record_sets()` method to get all record set IDs, and inspect fields for a selected record set.


In [ ]:
# List all available record sets and their @id
record_sets = dataset.list_record_sets()
print("Record Sets available in the dataset (by @id):")
for idx, rec in enumerate(record_sets):
    print(f"{idx+1}. {rec}")

# For demonstration, pick the first record set
if record_sets:
    primary_record_set = record_sets[0]
    print(f"\nInspecting structure of record set: {primary_record_set}")
    record_set_obj = dataset.get_record_set(primary_record_set)
    print("Fields (@id and name):")
    for fld in record_set_obj.fields:
        print(f" - {fld['@id']} (name: {fld.get('name', 'N/A')})")
else:
    print("No record sets found in the dataset.")

### Example records from the first record set

Let us examine a few records by their field `@id` from the dataset.

In [ ]:
# Print first three records from the primary record set
N = 3
if record_sets:
    for i, rec in enumerate(dataset.records(record_set=primary_record_set)):
        print(f"--- Record {i+1} ---")
        for k, v in rec.items():
            print(f"{k}: {v}")
        if i+1 >= N:
            break
else:
    print("No record sets to extract records from.")

## 3. Data Extraction

Load data from the main record sets into **DataFrames** for analysis. We use the list of record set IDs found in the previous step. Fields, columns, and other entities are always referenced by their `@id`.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = dict()
for rec_id in record_sets:
    print(f"Loading records for record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Record set '{rec_id}' -- {len(df)} records, fields: {df.columns.tolist()}")
        dataframes[rec_id] = df
    else:
        print(f"No records found for {rec_id}.")

# For demonstration, show column names and first few rows from the primary record set
if primary_record_set in dataframes:
    df = dataframes[primary_record_set]
    print(f"\nColumns in DataFrame for {primary_record_set}:")
    print(df.columns.tolist())
    df.head()
else:
    print(f"No data for {primary_record_set}.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalization, and grouping. We'll select a **numeric field** (e.g., `PHQ-9 score` if present) by its `@id` and a **grouping field** (e.g., gender or education).

In [ ]:
import numpy as np

# Identify good candidate field @ids automatically: look for common numeric fields in first record set
if primary_record_set in dataframes:
    df = dataframes[primary_record_set]
    # Try to pick a numeric survey score field by @id (usually lower-case, e.g., 'phq9', 'gad7', or similar)
    candidate_numeric_fields = [c for c in df.columns if ('phq' in c.lower() or 'gad' in c.lower() or 'score' in c.lower())]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=[np.number]).columns[0]
        print(f"Default numeric field: {numeric_field}")
else:
    print(f"No dataframe for record set {primary_record_set}")
    numeric_field = None

# Select a grouping field e.g. for gender/education if present
group_fields = [f for f in df.columns if any(x in f.lower() for x in ['gender', 'sex', 'education', 'marital', 'village'])]
group_field = group_fields[0] if group_fields else None
if group_field:
    print(f"Using field '{group_field}' for grouping.")
else:
    print("No suitable group field found, skipping grouping step.")

# Drop rows where numeric_field is missing
filtered_df = df.dropna(subset=[numeric_field])

# Define a threshold for analysis (e.g., 10)
threshold = 10
filt = filtered_df[numeric_field] > threshold
filtered_above_thresh = filtered_df[filt]
print(f"Number of records with {numeric_field} > {threshold}: {filtered_above_thresh.shape[0]}")

# Normalize the selected numeric field
filtered_above_thresh[f"{numeric_field}_normalized"] = (
    filtered_above_thresh[numeric_field] - filtered_above_thresh[numeric_field].mean()
) / filtered_above_thresh[numeric_field].std()

print(filtered_above_thresh[[numeric_field, f"{numeric_field}_normalized"]].head())

# If group_field is found, group by and report mean
if group_field:
    grouped_df = filtered_above_thresh.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field], bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field} in the dataset")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field if present
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- **Dataset Fields:** The dataset provides detailed survey records including demographics and mental health scales (e.g., GAD-7, PHQ-9). All field references are by Croissant `@id`.
- **Data Processing:** The notebook demonstrates basic EDA—filtering for high scores, normalization, and examining groupwise mean values.
- **Visualization:** Distributions of selected scores help reveal data skew and possible grouping effects by demographics.

For further work, consider handling missing values, exploring additional survey scales, or training predictive models.